In [1]:
%load_ext autoreload

In [8]:
%autoreload 2
from __future__ import annotations

import logging
from pathlib import Path

import polars as pl

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
)

from src.loading import get_df_by_prefix, load_csvs_to_polars
from src.cleaning import prepare_lands, prepare_projects, prepare_rents, prepare_transactions
from src.aggregation import (
    aggregate_rents,
    aggregate_sales,
    combine_sales_rents,
    enrich_rents,
    enrich_transactions,
    make_project_dimension,
)

In [9]:
data_dir = "data"

dataframes = load_csvs_to_polars(data_dir)

projects = prepare_projects(get_df_by_prefix(dataframes, "projects-"))
lands = prepare_lands(get_df_by_prefix(dataframes, "lands-"))
transactions = prepare_transactions(get_df_by_prefix(dataframes, "transactions-"))
rents = prepare_rents(get_df_by_prefix(dataframes, "rents-"))

INFO | src.cleaning | projects: starting cleaning (187 rows)
INFO | src.cleaning | projects | START_DATE: all non-null values converted successfully
INFO | src.cleaning | projects | END_DATE: all non-null values converted successfully
INFO | src.cleaning | projects | ADOPTION_DATE: all non-null values converted successfully
INFO | src.cleaning | projects | INSPECTION_DATE: all non-null values converted successfully
INFO | src.cleaning | projects | COMPLETION_DATE: all non-null values converted successfully
INFO | src.cleaning | projects | PROJECT_VALUE: all non-null values converted successfully
INFO | src.cleaning | projects | PERCENT_COMPLETED: all non-null values converted successfully
INFO | src.cleaning | projects | CNT_LAND: all non-null values converted successfully
INFO | src.cleaning | projects | CNT_BUILDING: all non-null values converted successfully
INFO | src.cleaning | projects | CNT_VILLA: all non-null values converted successfully
INFO | src.cleaning | projects | CNT_UN

In [10]:
project_land_dim = make_project_dimension(projects, lands)
transactions_enriched = enrich_transactions(transactions, project_land_dim)
rents_enriched = enrich_rents(rents, project_land_dim)

In [11]:
seg_property_cols = ["PROP_TYPE_EN"]
seg_developer_cols = ["DEVELOPER_EN"]
seg_land_cols = ["LAND_TYPE_EN"]

performance_by_property = combine_sales_rents(
    aggregate_sales(transactions_enriched, seg_property_cols),
    aggregate_rents(rents_enriched, seg_property_cols),
    seg_property_cols,
)
performance_by_developer = combine_sales_rents(
    aggregate_sales(transactions_enriched, seg_developer_cols),
    aggregate_rents(rents_enriched, seg_developer_cols),
    seg_developer_cols,
)
performance_by_land = combine_sales_rents(
    aggregate_sales(transactions_enriched, seg_land_cols),
    aggregate_rents(rents_enriched, seg_land_cols),
    seg_land_cols,
)

In [12]:
output_dir = "outputs"
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

outputs = {
    "transactions_enriched": transactions_enriched,
    "rents_enriched": rents_enriched,
    "performance_by_property_type": performance_by_property,
    "performance_by_developer": performance_by_developer,
    "performance_by_land_type": performance_by_land,
}

for name, df in outputs.items():
    df.write_csv(output_path / f"{name}.csv")

In [ ]:
projects_by_developer = (
    projects
    .group_by("DEVELOPER_EN")
    .agg(
        pl.len().alias("total"),
        (pl.col("PROJECT_STATUS") == "ACTIVE").sum().alias("ongoing"),
        (pl.col("PROJECT_STATUS") == "PENDING").sum().alias("planned"),
    )
    .sort("total", descending=True)
)

projects_by_developer

DEVELOPER_EN,total,ongoing,planned
str,u32,u32,u32
"""Binghatti Developers FZE""",12,8,0
"""EMAAR DEVELOPMENT P.J.S.C.""",10,10,0
"""AZIZI DEVELOPMENTS L.L.C""",8,4,0
"""AURORA SPV 3 L.L.C""",4,4,0
"""DUBAI SOUTH PROPERTIES DWC LLC""",4,4,0
…,…,…,…
"""LUNAYA ZAYA REAL ESTATE DEVELO…",1,0,0
"""AUM 99 HOME REAL ESTATE DEVELO…",1,1,0
"""K M Y REAL ESTATE L.L.C""",1,0,0
